#### Gesamtvergleich aller Methoden

Liest die bereits gespeicherten `_summary.json`-Dateien aller Methoden-Notebooks ein (Laplace Seed 2/Seed 1, MFVI Seed 2/Seed 1, Multi-Chain MILE K=3, Deep Ensemble N=3) -- rechnet nichts neu. Einzige Ausnahme: die Baseline wird einmalig als "1-Sample-Ensemble" ueber dieselbe Pipeline ausgewertet, damit ihre Metriken exakt dieselben Formeln nutzen wie alle anderen Methoden (statt die eigenen `test_metrics` aus `evaluate_saved_baseline.py` zu uebernehmen, die evtl. anders definiert sind).

Ersetzt die veraltete Version dieses Notebooks (referenzierte noch die 128er/512er-Configs ohne ECE, Multi-Chain-MILE oder Deep Ensemble).

In [1]:
from pathlib import Path

# ============================================================
# CONFIG
# GESAMTVERGLEICH ALLER METHODEN
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)

RESULTS_ROOT = MASTER_DIR / "method_results"
COMPARISON_RESULT_PATH = RESULTS_ROOT / "all_methods_comparison.json"

EVAL_BATCH_SIZE = 64

# ------------------------------------------------------------
# Gespeicherte Runs -- liest NUR die bereits vorhandenen
# _summary.json-Dateien, rechnet nichts neu (Baseline ausgenommen,
# siehe unten). Passe run_name an, falls ein Methoden-Notebook mit
# anderer Config neu gelaufen ist.
# ------------------------------------------------------------

RUNS = [
    ("Laplace (Seed 2, Haupt)", RESULTS_ROOT / "laplace_qwen_agnews",
     "laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10"),
    ("Laplace (Seed 1)", RESULTS_ROOT / "laplace_qwen_agnews",
     "laplace_seed1_balanced2048_seq32_prior0.02_fscale1.0_samples10"),
    ("MFVI (Seed 2, Haupt)", RESULTS_ROOT / "mfvi_qwen_agnews",
     "mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval"),
    ("MFVI (Seed 1)", RESULTS_ROOT / "mfvi_qwen_agnews",
     "mfvi_seed1_balanced2048_seq32_prior0.02_steps200_samples10_testeval"),
    ("MILE (Multi-Chain, K=3)", RESULTS_ROOT / "mile_qwen_agnews",
     "mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval"),
    ("Deep Ensemble (N=3)", RESULTS_ROOT / "deep_ensemble_qwen_agnews",
     "deep_ensemble_n3_seq32"),
]

print("Konfigurierte Runs:", len(RUNS))
for label, result_dir, run_name in RUNS:
    print(" -", label, "->", run_name)


Konfigurierte Runs: 6
 - Laplace (Seed 2, Haupt) -> laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10
 - Laplace (Seed 1) -> laplace_seed1_balanced2048_seq32_prior0.02_fscale1.0_samples10
 - MFVI (Seed 2, Haupt) -> mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval
 - MFVI (Seed 1) -> mfvi_seed1_balanced2048_seq32_prior0.02_steps200_samples10_testeval
 - MILE (Multi-Chain, K=3) -> mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval
 - Deep Ensemble (N=3) -> deep_ensemble_n3_seq32


In [2]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import json
import runpy

import numpy as np
import jax
import jax.numpy as jnp
from jax import random

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3
Devices: [CudaDevice(id=0)]


In [3]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [4]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (Seed 2)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246214
val_acc: 0.913667
val_ece: 0.009068
val_brier: 0.131597
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [5]:
# ============================================================
# REBUILD_FULL_PARAMS (LoRA-Struktur -- identisch fuer alle Methoden)
# ============================================================

theta_map, all_leaves, lora_indices, full_tree_definition, unravel_lora = (
    qpu.vectorize_lora_params(params["params"])
)

rebuild_full_params = qpu.build_rebuild_fn(
    all_leaves, lora_indices, full_tree_definition, unravel_lora,
)

print("theta_map shape:", theta_map.shape)


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
theta_map shape: (540672,)


In [6]:
# ============================================================
# BASELINE ALS "1-SAMPLE-ENSEMBLE" -- gleiche Pipeline/Metrikdefinitionen
# wie alle anderen Methoden, damit der Vergleich exakt konsistent ist.
# Mit nur einem Sample ist MI per Definition 0 (keine Uneinigkeit ohne
# zweites Sample moeglich) und Predictive Entropy == Expected Entropy
# (die gesamte Unsicherheit ist "aleatorisch" -- es gibt kein Ensemble,
# das sich widersprechen koennte).
# ============================================================

x_id, y_id = data.get("test")
print("Testbeispiele:", int(y_id.shape[0]))

baseline_theta = np.asarray(jax.device_get(theta_map))[None, :]

baseline_sample_probabilities, baseline_mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=baseline_theta,
        x_eval=x_id,
        y_eval=y_id,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)

baseline_metrics = qpu.compute_predictive_metrics(
    sample_probabilities=baseline_sample_probabilities,
    mean_probabilities=baseline_mean_probabilities,
    y_true=y_id,
)

baseline_ece = qpu.multiclass_ece(baseline_mean_probabilities, y_id, n_bins=15)

baseline_summary, _ = qpu.summarize_metrics(baseline_metrics)
baseline_summary["ece"] = float(baseline_ece)
baseline_summary["run_name"] = "baseline_seed2"
baseline_summary["method"] = "Baseline (deterministisch)"
baseline_summary["label"] = "Baseline (Seed 2)"
baseline_summary["seed"] = 2

print()
print("Baseline-Metriken (ueber dieselbe Pipeline wie alle Methoden):")
for key in ["accuracy", "posterior_predictive_nll", "brier_score", "ece",
            "mean_predictive_entropy", "mean_expected_entropy", "mean_mutual_information"]:
    print(f"  {key}: {baseline_summary[key]}")


Testbeispiele: 7600
Evaluation examples: 7600
Evaluation batch size: 64
Posterior samples: 1
Evaluating posterior sample 1/1

Evaluation completed
Probability tensor shape: (1, 7600, 4)
Expected shape: (1, 7600, 4)
All probabilities finite: True
Accuracy: 0.914736807346344
LPPD: -0.2510938048362732
Posterior predictive NLL: 0.2510938048362732
Brier Score: 0.13155591487884521
Mean predictive entropy: 0.23771847784519196
Mean expected entropy: 0.23771847784519196
Mean mutual information: 0.0

Baseline-Metriken (ueber dieselbe Pipeline wie alle Methoden):
  accuracy: 0.914736807346344
  posterior_predictive_nll: 0.2510938048362732
  brier_score: 0.13155591487884521
  ece: 0.0075426893308758736
  mean_predictive_entropy: 0.23771850764751434
  mean_expected_entropy: 0.23771850764751434
  mean_mutual_information: 0.0


In [7]:
# ============================================================
# GESPEICHERTE SUMMARIES LADEN (kein Neuberechnen -- nur Einlesen)
# ============================================================

loaded_summaries = []

for label, result_dir, run_name in RUNS:
    summary_path = Path(result_dir) / f"{run_name}_summary.json"

    if not summary_path.exists():
        print(f"UEBERSPRUNGEN (nicht gefunden): {label} -> {summary_path}")
        continue

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    summary["label"] = label
    loaded_summaries.append(summary)
    print(f"Geladen: {label} ({run_name})")

print()
print(len(loaded_summaries), "von", len(RUNS), "Runs gefunden.")


Geladen: Laplace (Seed 2, Haupt) (laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10)
Geladen: Laplace (Seed 1) (laplace_seed1_balanced2048_seq32_prior0.02_fscale1.0_samples10)
Geladen: MFVI (Seed 2, Haupt) (mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval)
Geladen: MFVI (Seed 1) (mfvi_seed1_balanced2048_seq32_prior0.02_steps200_samples10_testeval)
Geladen: MILE (Multi-Chain, K=3) (mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval)
Geladen: Deep Ensemble (N=3) (deep_ensemble_n3_seq32)

6 von 6 Runs gefunden.


In [8]:
# ============================================================
# OOD-ERGEBNISSE EINBINDEN (aus 07_ood_evaluation.ipynb, per
# run_name gematcht) -- falls die Datei noch nicht existiert (07
# noch nicht gelaufen), wird das nur vermerkt, kein Fehler. Beim
# erneuten Ausfuehren dieser Zelle nach einem 07-Lauf werden die
# OOD-Spalten automatisch nachgezogen.
# ============================================================

OOD_RESULT_PATH = RESULTS_ROOT / "ood_imdb_comparison.json"

ood_by_run_name = {}

if OOD_RESULT_PATH.exists():
    with open(OOD_RESULT_PATH, "r", encoding="utf-8") as f:
        ood_data = json.load(f)
    for entry in ood_data["results"]:
        ood_by_run_name[entry["run_name"]] = entry
    print(f"OOD-Vergleich geladen: {len(ood_by_run_name)} Methoden ({OOD_RESULT_PATH})")
else:
    print(f"OOD-Vergleich noch nicht vorhanden ({OOD_RESULT_PATH})")
    print("-- 07_ood_evaluation.ipynb erst noch laufen lassen, dann diese Zelle neu ausfuehren.")

for row in loaded_summaries:
    ood_entry = ood_by_run_name.get(row.get("run_name"))
    if ood_entry is not None:
        row["ood_mean_mi"] = ood_entry["ood_mean_mi"]
        row["auroc_mutual_information"] = ood_entry["auroc_mutual_information"]
        row["auroc_predictive_entropy"] = ood_entry["auroc_predictive_entropy"]

n_matched = sum(1 for row in loaded_summaries if "auroc_mutual_information" in row)
print(f"\n{n_matched} von {len(loaded_summaries)} geladenen Runs haben passende OOD-Ergebnisse.")


OOD-Vergleich geladen: 4 Methoden (/dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/ood_imdb_comparison.json)

4 von 6 geladenen Runs haben passende OOD-Ergebnisse.


In [9]:
# ============================================================
# MASTER-VERGLEICHSTABELLE (inkl. OOD-Spalten, falls 07 gelaufen ist)
# ============================================================

all_rows = [baseline_summary] + loaded_summaries

columns = [
    ("label", "Methode", 26),
    ("accuracy", "Accuracy", 9),
    ("posterior_predictive_nll", "NLL", 8),
    ("brier_score", "Brier", 8),
    ("ece", "ECE", 8),
    ("mean_predictive_entropy", "Pred.Ent.", 10),
    ("mean_expected_entropy", "Exp.Ent.", 10),
    ("mean_mutual_information", "MI", 8),
    ("auroc_mutual_information", "AUROC(MI)", 10),
    ("auroc_predictive_entropy", "AUROC(Ent.)", 11),
]


def epistemic_fraction(row):
    pred_ent = row.get("mean_predictive_entropy")
    mi = row.get("mean_mutual_information")
    if not pred_ent:
        return None
    return mi / pred_ent


def format_cell(value, width):
    if value is None:
        return "n/a".rjust(width)
    if isinstance(value, float):
        return f"{value:.4f}".rjust(width)
    return str(value).rjust(width)


header = "  ".join(name.rjust(width) for _, name, width in columns) + "  " + "Epist.%".rjust(8)
print(header)
print("-" * len(header))

for row in all_rows:
    parts = [format_cell(row.get(key), width) for key, _, width in columns]
    frac = epistemic_fraction(row)
    frac_text = f"{frac * 100:.1f}%" if frac is not None else "n/a"
    print("  ".join(parts) + "  " + frac_text.rjust(8))

print()
print("AUROC-Spalten sind 'n/a', solange 07_ood_evaluation.ipynb noch nicht fuer die jeweilige")
print("Methode gelaufen ist (oder die vorherige Zelle noch keine OOD-Datei gefunden hat).")


                   Methode   Accuracy       NLL     Brier       ECE   Pred.Ent.    Exp.Ent.        MI   AUROC(MI)  AUROC(Ent.)   Epist.%
----------------------------------------------------------------------------------------------------------------------------------------
         Baseline (Seed 2)     0.9147    0.2511    0.1316    0.0075      0.2377      0.2377    0.0000         n/a          n/a      0.0%
   Laplace (Seed 2, Haupt)     0.9093    0.3116    0.1534    0.0753      0.4881      0.4305    0.0576      0.7404       0.9452     11.8%
          Laplace (Seed 1)     0.9066    0.3254    0.1595    0.0931      0.5470      0.4821    0.0649         n/a          n/a     11.9%
      MFVI (Seed 2, Haupt)     0.8896    0.8268    0.1939    0.0851      0.0592      0.0374    0.0218      0.8813       0.9662     36.8%
             MFVI (Seed 1)     0.8878    0.7998    0.1946    0.0866      0.0612      0.0358    0.0254         n/a          n/a     41.5%
   MILE (Multi-Chain, K=3)     0.9083    

In [10]:
# ============================================================
# SEED-ROBUSTHEITS-VERGLEICH (Seed 2 Hauptlauf vs. Seed 1 Zusatzlauf)
# ============================================================


def find_row(label_substring):
    for row in loaded_summaries:
        if label_substring in row["label"]:
            return row
    return None


for method_name in ["Laplace", "MFVI"]:
    seed2_row = find_row(f"{method_name} (Seed 2")
    seed1_row = find_row(f"{method_name} (Seed 1")
    if seed2_row is None or seed1_row is None:
        print(f"{method_name}: einer der beiden Seed-Laeufe fehlt noch.")
        continue
    print(f"\n=== {method_name}: Seed 2 vs. Seed 1 ===")
    for key, name in [
        ("accuracy", "Accuracy"), ("posterior_predictive_nll", "NLL"),
        ("brier_score", "Brier"), ("ece", "ECE"),
        ("mean_mutual_information", "Mean MI"),
    ]:
        v2 = seed2_row.get(key)
        v1 = seed1_row.get(key)
        diff = (v1 - v2) if (v1 is not None and v2 is not None) else None
        diff_text = f"(Diff: {diff:+.4f})" if diff is not None else ""
        print(f"  {name:10s}  Seed2={v2:.4f}   Seed1={v1:.4f}  {diff_text}")



=== Laplace: Seed 2 vs. Seed 1 ===
  Accuracy    Seed2=0.9093   Seed1=0.9066  (Diff: -0.0028)
  NLL         Seed2=0.3116   Seed1=0.3254  (Diff: +0.0139)
  Brier       Seed2=0.1534   Seed1=0.1595  (Diff: +0.0061)
  ECE         Seed2=0.0753   Seed1=0.0931  (Diff: +0.0178)
  Mean MI     Seed2=0.0576   Seed1=0.0649  (Diff: +0.0074)

=== MFVI: Seed 2 vs. Seed 1 ===
  Accuracy    Seed2=0.8896   Seed1=0.8878  (Diff: -0.0018)
  NLL         Seed2=0.8268   Seed1=0.7998  (Diff: -0.0270)
  Brier       Seed2=0.1939   Seed1=0.1946  (Diff: +0.0007)
  ECE         Seed2=0.0851   Seed1=0.0866  (Diff: +0.0016)
  Mean MI     Seed2=0.0218   Seed1=0.0254  (Diff: +0.0037)


In [11]:
# ============================================================
# LAUFZEITEN-UEBERSICHT (soweit in den jeweiligen Summaries erfasst)
# ============================================================

print("Laufzeiten (soweit gemessen):\n")

for row in loaded_summaries:
    label = row["label"]
    parts = []
    if "fisher_duration_seconds" in row:
        parts.append(f"Fisher-Berechnung: {row['fisher_duration_seconds']:.1f}s")
    if "final_training_duration_seconds" in row:
        parts.append(f"Training: {row['final_training_duration_seconds']:.1f}s")
    if "chains_total_duration_seconds" in row:
        parts.append(f"Chains gesamt: {row['chains_total_duration_seconds']:.1f}s")
    if "chain_durations_seconds" in row:
        parts.append(f"Chains einzeln: {row['chain_durations_seconds']}")
    if "notebook_total_duration_seconds" in row:
        total = row["notebook_total_duration_seconds"]
        parts.append(f"Notebook gesamt: {total:.1f}s ({total / 60:.1f} min)")

    if parts:
        print(f"{label}:")
        for p in parts:
            print(f"  {p}")
    else:
        print(f"{label}: keine Laufzeitdaten erfasst")
    print()


Laufzeiten (soweit gemessen):

Laplace (Seed 2, Haupt): keine Laufzeitdaten erfasst

Laplace (Seed 1):
  Fisher-Berechnung: 147.0s
  Notebook gesamt: 2328.3s (38.8 min)

MFVI (Seed 2, Haupt): keine Laufzeitdaten erfasst

MFVI (Seed 1):
  Training: 1036.9s

MILE (Multi-Chain, K=3):
  Chains gesamt: 0.1s
  Chains einzeln: {'1': 7855.551316738129, '2': 7918.149289131165, '3': 7927.052138805389}
  Notebook gesamt: 10157.0s (169.3 min)

Deep Ensemble (N=3): keine Laufzeitdaten erfasst



In [12]:
# ============================================================
# GESAMTVERGLEICH SPEICHERN
# ============================================================

output = {
    "baseline": baseline_summary,
    "methods": loaded_summaries,
}

with open(COMPARISON_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, default=str)

print("Gespeichert:", COMPARISON_RESULT_PATH)


Gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/all_methods_comparison.json
